In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("1. Membaca Data Bersih...")
# Kita pakai data hasil kerjaan Data Engineer lo kemarin!
df = pd.read_parquet('"D:/PROJECT/DE/data/processed/wind_turbine_cleaned.parquet')

# Trik Data Science: Kita cuma mau memprediksi daya saat mesin beroperasi NORMAL.
# Kalau mesin lagi error, nggak usah diprediksi (karena pasti dayanya 0).
df_normal = df[df['turbine_status'] == 'Normal']

print("2. Memilih Fitur (X) dan Target (y)...")
# X (Fitur) = Data yang kita jadikan petunjuk/input
X = df_normal[['wind_speed_ms', 'wind_direction_deg']]

# y (Target) = Data yang mau kita tebak/prediksi
y = df_normal['active_power_kw']

print("3. Memecah Data (Train-Test Split)...")
# Kita pisahkan 80% data untuk "belajar" (Train) dan 20% untuk "ujian" (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("4. Melatih Model (Training)...")
# Kita pakai algoritma Random Forest (kumpulan pohon keputusan)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("5. Ujian! Mengevaluasi Akurasi Model...")
# Kita suruh model menebak X_test, lalu bandingkan hasilnya dengan y_test (kunci jawaban asli)
prediksi = model.predict(X_test)

# Hitung error-nya
rmse = np.sqrt(mean_squared_error(y_test, prediksi))
r_squared = r2_score(y_test, prediksi)

print("-" * 30)
print(f"Beda tebakan dengan aslinya (RMSE): {rmse:.2f} kW")
print(f"Skor Akurasi Model (R-Squared): {r_squared * 100:.2f}%")
print("-" * 30)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Mengatur tema agar grafik terlihat profesional
sns.set_theme(style="whitegrid")

# Menentukan ukuran kanvas
plt.figure(figsize=(10, 6))

# 1. Membuat titik-titik persebaran data (Asli vs Prediksi)
# Kita pakai parameter 'alpha' agar titik yang menumpuk terlihat lebih gelap
plt.scatter(y_test, prediksi, alpha=0.3, color='dodgerblue', label='Hasil Prediksi')

# 2. Membuat garis diagonal merah sebagai batas "Prediksi Sempurna"
# Jika tebakan AI 100% sama dengan aslinya, semua titik biru akan jatuh pas di garis merah ini
max_val = max(y_test.max(), prediksi.max())
plt.plot([0, max_val], [0, max_val], color='crimson', linestyle='--', linewidth=2, label='Prediksi Sempurna (Akurat 100%)')

# 3. Menambahkan judul dan label
plt.title('Evaluasi Model AI: Daya Listrik Aktual vs Prediksi', fontsize=14, fontweight='bold')
plt.xlabel('Daya Aktual Asli (kW)', fontsize=12)
plt.ylabel('Daya Tebakan AI (kW)', fontsize=12)
plt.legend(fontsize=11)

# Menampilkan grafik
plt.tight_layout()
plt.show()

In [ ]:
# Mengambil nilai seberapa penting setiap fitur dari model Random Forest
importances = model.feature_importances_
nama_fitur = X.columns

# Memasukkan datanya ke dalam tabel (DataFrame) biar rapi
df_importance = pd.DataFrame({'Fitur': nama_fitur, 'Tingkat Kepentingan': importances})
df_importance = df_importance.sort_values(by='Tingkat Kepentingan', ascending=False)

# Menggambar grafiknya
plt.figure(figsize=(8, 4))
sns.barplot(x='Tingkat Kepentingan', y='Fitur', data=df_importance, palette='magma')

plt.title('Feature Importance: Faktor Penentu Daya Listrik', fontsize=14, fontweight='bold')
plt.xlabel('Skor Kepentingan (0 - 1.0)', fontsize=12)
plt.ylabel('')
plt.tight_layout()
plt.show()